In [1]:
# ── Cell 1 · Load regime-mapped test dataset ─────────────────────────────────
import pandas as pd
import numpy as np
from pathlib import Path

def find_project_root(marker=".git"):
    p = Path.cwd()
    while p != p.parent:
        if (p / marker).exists():
            return p
        p = p.parent
    raise FileNotFoundError("Project root not found")

DATA_PATH = find_project_root() / "data" / "mlData" / "trainData" / "202603-vX-test-regime-mapped.jsonl"

df = pd.read_json(DATA_PATH, lines=True, convert_dates=False)
df["dt"] = pd.to_datetime(df["timestamp"], unit="ms")

print(f"Rows   : {len(df):,}")
print(f"Period : {df['dt'].iloc[0]} → {df['dt'].iloc[-1]}")
print(f"Labels : {sorted(df['label'].unique())}")
print(f"Regimes: {sorted(df['regime'].unique())}")

Rows   : 54,589
Period : 2025-09-02 11:00:00 → 2026-03-11 00:00:00
Labels : [np.int64(-1), np.int64(0), np.int64(1)]
Regimes: ['ACCUMULATION', 'BEARISH', 'BULLISH', 'CORRECTION', 'DISTRIBUTION', 'RECOVERY']


In [2]:
# ── Cell 2 · Label distribution per regime (table) ───────────────────────────
REGIME_ORDER = ["ACCUMULATION", "BULLISH", "RECOVERY",
                "DISTRIBUTION", "BEARISH",  "CORRECTION", "UNKNOWN", "UNKNOWN_WARMUP"]
LABEL_VALUES = [-1, 0, 1]

rows = []
for regime in REGIME_ORDER:
    sub = df[df["regime"] == regime]
    if len(sub) == 0:
        continue
    counts = sub["label"].value_counts()
    total  = len(sub)
    row    = {"regime": regime, "total": total}
    for lv in LABEL_VALUES:
        c = counts.get(lv, 0)
        row[f"label_{lv}_n"]   = c
        row[f"label_{lv}_pct"] = 100 * c / total
    rows.append(row)

dist = pd.DataFrame(rows).set_index("regime")

header = (f"{'Regime':<18}  {'Total':>7}  "
          f"{'L=-1 (n)':>9}  {'%':>6}  "
          f"{'L=0 (n)':>8}  {'%':>6}  "
          f"{'L=+1 (n)':>9}  {'%':>6}")
print(header)
print("-" * len(header))
for r in REGIME_ORDER:
    if r not in dist.index:
        continue
    d = dist.loc[r]
    print(
        f"{r:<18}  {int(d['total']):>7,}  "
        f"{int(d['label_-1_n']):>9,}  {d['label_-1_pct']:>5.1f}%  "
        f"{int(d['label_0_n']):>8,}  {d['label_0_pct']:>5.1f}%  "
        f"{int(d['label_1_n']):>9,}  {d['label_1_pct']:>5.1f}%"
    )

Regime                Total   L=-1 (n)       %   L=0 (n)       %   L=+1 (n)       %
-----------------------------------------------------------------------------------
ACCUMULATION         13,207      4,649   35.2%     4,251   32.2%      4,307   32.6%
BULLISH               8,337      2,719   32.6%     2,733   32.8%      2,885   34.6%
RECOVERY              1,029        297   28.9%       406   39.5%        326   31.7%
DISTRIBUTION          7,206      2,296   31.9%     2,504   34.7%      2,406   33.4%
BEARISH              21,573      7,275   33.7%     7,091   32.9%      7,207   33.4%
CORRECTION            3,237      1,093   33.8%     1,154   35.7%        990   30.6%


In [3]:
# ── Cell 3 · Stacked bar chart ────────────────────────────────────────────────
import plotly.graph_objects as go

regime_labels = [r for r in REGIME_ORDER if r in dist.index]
LABEL_COLORS  = {-1: "#EF5350", 0: "#B0BEC5", 1: "#66BB6A"}
LABEL_NAMES   = {-1: "Short (-1)", 0: "Neutral (0)", 1: "Long (+1)"}

fig = go.Figure()
for lv in LABEL_VALUES:
    pcts = [dist.loc[r, f"label_{lv}_pct"] for r in regime_labels]
    ns   = [int(dist.loc[r, f"label_{lv}_n"]) for r in regime_labels]
    fig.add_trace(go.Bar(
        name=LABEL_NAMES[lv],
        x=regime_labels,
        y=pcts,
        marker_color=LABEL_COLORS[lv],
        text=[f"{p:.1f}%<br>(n={n:,})" for p, n in zip(pcts, ns)],
        textposition="inside",
        insidetextanchor="middle",
    ))

fig.update_layout(
    barmode="stack",
    title="<b>Label Distribution per Regime — 202603-vX Test Set</b>",
    xaxis_title="Regime",
    yaxis=dict(title="% of bars", range=[0, 100]),
    legend=dict(orientation="h", y=-0.18),
    height=480,
    plot_bgcolor="#FAFAFA",
    paper_bgcolor="#FFFFFF",
)
fig.show()